In [1]:
!pip install sentencepiece tiktoken evaluate
!pip install --upgrade transformers

In [2]:
import pandas as pd
from google.colab import drive
from transformers import AutoTokenizer, BartForConditionalGeneration, BartTokenizer, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer, AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import Dataset
import torch
import evaluate

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
splits = ['Train', 'Val', 'Test']
dfs = {}

for split in splits:
  path = f'/content/drive/MyDrive/266_Final/data/{split}/wikidt_{split.lower()}_tabular.parquet'
  dfs[split] = pd.read_parquet(path)
  print(f'{split} shape: {dfs[split].shape}')

train_df, val_df, test_df = dfs['Train'], dfs['Val'], dfs['Test']

Train shape: (53492, 10)
Val shape: (6686, 10)
Test shape: (6687, 10)


In [5]:
train_df.head()

,question,answer,context,pairs,positive_sentences,image,context_tables_json,num_tables,context_multitabqa,context_tapex
0,geno hayes and nick roach both played which po...,no answer,Position Player $ Free agency tag Date signed ...,"[{'label': 0, 'query': 'geno hayes and nick ro...",0,204_783_page.png,"{""tables"":[{""table_name"":""free_agents"",""header...",3,free_agents : col : position | player | $ | fr...,free_agents : col : position | player | $ | fr...
1,What is the number of bronze for Canada and si...,no answer,"Pos Team Time Gold Switzerland (Reto Götschi, ...","[{'label': 0, 'query': 'What is the number of ...",0,11338408_page.png,"{""tables"":[{""table_name"":""two_man_bobsleigh"",""...",4,two_man_bobsleigh : col : pos | team | time | ...,two_man_bobsleigh : col : pos | team | time | ...
2,"What is Date, when Opponent is ""Philadelphia 7...",february 16,# Date H/A/N Opponent W/L Score Record 40 Janu...,"[{'label': 1, 'query': 'What is Date, when Opp...",1,16946097_page.png,"{""tables"":[{""table_name"":""game_results"",""heade...",1,game_results : col : date | h/a/n | opponent |...,col : date | h/a/n | opponent | w/l | score | ...
3,Which Class has a Weight of 203?,sr,Position Number Name Height Weight Class Homet...,"[{'label': 1, 'query': 'Which Class has a Weig...",29,14624447_page.png,"{""tables"":[{""table_name"":""defensive_players"",""...",6,defensive_players : col : position | number | ...,defensive_players : col : position | number | ...
4,Who had 0 total votes in the purple team?,eliminated week 12 - the walk,Members Teammate Couples Team Blue Red 1 Blue ...,"[{'label': 1, 'query': 'Who had 0 total votes ...",1,18426579_page.png,"{""tables"":[{""table_name"":""castaway_elimination...",1,castaway_eliminations : col : members | teamma...,col : members | teammate | couples | team | bl...


In [6]:
train_df['context'][45000]

'Seat State Majority Member Party HINKLER Qld 00.0 Paul Neville NPA SOLOMON NT 00.1 Dave Tollner Lib ADELAIDE SA 00.2 Hon Trish Worth Lib CANNING WA 00.4 Don Randall Lib DOBELL NSW 00.4 Ken Ticehurst Lib PARRAMATTA NSW 01.1 Ross Cameron Lib McEWEN Vic 01.2 Fran Bailey Lib PATERSON NSW 01.4 Bob Baldwin Lib HERBERT Qld 01.6 Peter Lindsay Lib RICHMOND NSW 01.6 Hon Larry Anthony NPA DEAKIN Vic 01.7 Philip Barresi Lib EDEN-MONARO NSW 01.7 Gary Nairn Lib HINDMARSH SA 01.9 Hon Christine Gallus Lib White (Democrat) 6,000 06.0% Smith (Labor) 45,000 45.0% Jones (Liberal) 35,000 35.0% Johnson (Green) 10,000 10.0% Davies (Ind) 4,000 04.0%'

In [7]:
model = "vaishali/multitabqa-base"

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(model)

config.json:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

In [9]:
def preprocess_function(examples):
  inputs = [
        f"{q.strip()} {c.strip()}"
        for q, c in zip(examples["question"], examples["context_multitabqa"])
  ]

  model_inputs = tokenizer(
        inputs,
        max_length=1024,
        padding=False,
        truncation=True,
    )

  labels = tokenizer(
        text_target=examples["answer"],
        max_length=1024,
        padding=False,
        truncation=True,
    )

  pad_id = tokenizer.pad_token_id
  model_inputs["labels"] = [
      [(token if token != pad_id else -100) for token in seq]
      for seq in labels["input_ids"]
  ]

  return model_inputs




In [10]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

In [11]:
tokenized_train_dataset = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_val_dataset = val_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=val_dataset.column_names,
)

tokenized_test_dataset = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names,
)

Map:   0%|          | 0/53492 [00:00<?, ? examples/s]

Map:   0%|          | 0/6686 [00:00<?, ? examples/s]

Map:   0%|          | 0/6687 [00:00<?, ? examples/s]

In [12]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

In [13]:
tokenized_train_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 53492
})

In [14]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./multitab-base-paper",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=32,
    eval_strategy="epoch",
    learning_rate=4e-5,
    num_train_epochs=10,
    predict_with_generate=True,
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    report_to="none",
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    data_collator=data_collator,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.515889
2,No log,0.475101
3,18.989264,0.477729
4,18.989264,0.461281
5,10.210756,0.470759
6,10.210756,0.478146
7,10.210756,0.492139
8,7.139876,0.506986
9,7.139876,0.514685
10,5.425953,0.520706


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2090, training_loss=10.20062047054893, metrics={'train_runtime': 2793.3738, 'train_samples_per_second': 191.496, 'train_steps_per_second': 0.748, 'total_flos': 3.254659007588352e+17, 'train_loss': 10.20062047054893, 'epoch': 10.0})

In [ ]:
test_output = trainer.predict(
    test_dataset=tokenized_test_dataset
)

In [ ]:
for k, v in test_output.metrics.items():
    print(f"{k}: {v}")

test_loss: 0.5328680276870728
test_runtime: 111.0686
test_samples_per_second: 60.206
test_steps_per_second: 7.527


In [ ]:
raw_preds = test_output.predictions

In [ ]:
predicted_answers = tokenizer.batch_decode(raw_preds, skip_special_tokens=True)

In [ ]:
test_results_df = pd.DataFrame({
    "question": test_df["question"].values,
    "context": test_df["context"].values,
    "ground_truth": test_df["answer"].astype(str).values,
    "predicted": [p.strip() for p in predicted_answers],
})

In [ ]:
test_results_df["is_exact_match"] = (
    test_results_df["predicted"].str.strip().str.lower()
    == test_results_df["ground_truth"].str.strip().str.lower()
)

In [ ]:
test_results_df[test_results_df["is_exact_match"] == True]

In [ ]:
test_results_df[test_results_df["is_exact_match"] == False]

In [ ]:
test_results_df[test_results_df["is_exact_match"] == False]['context'][6678]

In [ ]:
em_metric = evaluate.load("exact_match")
squad_metric = evaluate.load("squad")

results = em_metric.compute(
    predictions=test_df["answer"].astype(str).values,
    references=[p.strip() for p in predicted_answers],
    ignore_case=True,
    ignore_punctuation=True,
)
print(f"EM: {results['exact_match'] * 100:.2f}%")
preds = [
    {"id": str(i), "prediction_text": p.strip()}
    for i, p in enumerate(predicted_answers)
]
refs = [
    {"id": str(i), "answers": {"text": [str(a)], "answer_start": [0]}}
    for i, a in enumerate(test_df["answer"])
]
results = squad_metric.compute(predictions=preds, references=refs)
print(results)